# S1_Definición — Evaluación Sumativa 1

**MCDI501: Estadística Computacional para la Toma de Decisiones**  
**Proyecto:** Análisis estadístico de condiciones meteorológicas en Australia  
**Dataset:** WeatherAUS / Rain in Australia  
**Integrantes:** Enzo Pinilla, Claudio Alarcón y Luis Rodrigo Espinoza  
**Docente:** Dr. Jean Paul Maidana González  
**Repositorio:** https://github.com/sauriomac/mcdi501-estadistica-computacional-grupo2

## Propósito del notebook

Este cuaderno corresponde a la **Evaluación Sumativa 1** y se construye a partir de la Formativa 1 y de la retroalimentación del profesor. Incorpora las mejoras solicitadas por la rúbrica:

- revisión documentada de faltantes, duplicados e inconsistencias de dominio;
- clasificación completa y no redundante de variables;
- frecuencias absolutas y relativas con gráficos para variables categóricas;
- cuantificación de asimetría y curtosis;
- estimación puntual e intervalos de confianza para al menos tres variables numéricas;
- intervalo de Wilson para la proporción de `RainTomorrow = Yes`;
- dos pruebas de hipótesis vinculadas con la decisión de anticipar lluvia, con supuestos y tamaño del efecto;
- semilla fija y ejecución reproducible de principio a fin.


## 0. Configuración inicial y reproducibilidad

Se fija una semilla aleatoria para que cualquier muestreo o procedimiento aleatorio sea reproducible. Aunque la mayoría de los cálculos son determinísticos, declarar la semilla permite dejar trazabilidad metodológica.

In [1]:
# Librerías base
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# Semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)

# Configuración visual general
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 10

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

print(f"Semilla fijada: {SEED}")

Semilla fijada: 42


## 1. Preparación y carga de datos

Esta sección responde a la preparación solicitada en la Sumativa 1: carga del dataset, revisión de estructura, tipos de variables, valores faltantes, duplicados, inconsistencias y limpieza básica documentada.

In [2]:
# Ruta reproducible del dataset dentro del repositorio.
# Se prueban ubicaciones relativas habituales sin fijar rutas absolutas del equipo.
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
candidate_paths = []
for root in candidate_roots:
    candidate_paths.extend([
        root / "data" / "raw" / "weatherAUS.csv",
        root / "weatherAUS.csv",
    ])

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "No se encontró weatherAUS.csv. Debe ubicarse en data/raw/weatherAUS.csv "
        "respecto de la raíz del repositorio."
    )

PROJECT_ROOT = DATA_PATH.parent.parent.parent if DATA_PATH.parent.name == "raw" else DATA_PATH.parent

try:
    shown_path = DATA_PATH.relative_to(PROJECT_ROOT)
except ValueError:
    shown_path = DATA_PATH

print(f"Dataset encontrado en: {shown_path}")
df_raw = pd.read_csv(DATA_PATH)
print(f"Dimensiones originales: {df_raw.shape[0]:,} filas x {df_raw.shape[1]:,} columnas")
df_raw.head()


Dataset encontrado en: data/raw/weatherAUS.csv
Dimensiones originales: 145,460 filas x 23 columnas


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [3]:
# Estructura general del dataset
info_resumen = pd.DataFrame({
    "variable": df_raw.columns,
    "tipo_pandas": [str(df_raw[c].dtype) for c in df_raw.columns],
    "no_nulos": [df_raw[c].notna().sum() for c in df_raw.columns],
    "nulos": [df_raw[c].isna().sum() for c in df_raw.columns],
    "% nulos": [(df_raw[c].isna().mean() * 100) for c in df_raw.columns],
    "valores_unicos": [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
})

info_resumen.sort_values("% nulos", ascending=False).reset_index(drop=True)

,variable,tipo_pandas,no_nulos,nulos,% nulos,valores_unicos
0,Sunshine,float64,75625,69835,48.009762,145
1,Evaporation,float64,82670,62790,43.166506,358
2,Cloud3pm,float64,86102,59358,40.807095,10
3,Cloud9am,float64,89572,55888,38.421559,10
4,Pressure9am,float64,130395,15065,10.356799,546
5,Pressure3pm,float64,130432,15028,10.331363,549
6,WindDir9am,object,134894,10566,7.263853,16
7,WindGustDir,object,135134,10326,7.098859,16
8,WindGustSpeed,float64,135197,10263,7.055548,67
9,Humidity3pm,float64,140953,4507,3.098446,101


In [4]:
# Revisión de duplicados exactos
duplicados = df_raw.duplicated().sum()
porc_duplicados = duplicados / len(df_raw) * 100
print(f"Filas duplicadas exactas: {duplicados:,} ({porc_duplicados:.3f}%)")

Filas duplicadas exactas: 0 (0.000%)


In [5]:
# Limpieza básica: copia de trabajo, fecha a datetime, eliminación de duplicados exactos si existen.
df = df_raw.copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)

# Variables binarias codificadas para análisis numérico cuando corresponde
binary_map = {"No": 0, "Yes": 1}
df["RainToday_bin"] = df["RainToday"].map(binary_map)
df["RainTomorrow_bin"] = df["RainTomorrow"].map(binary_map)

print(f"Dimensiones tras limpieza básica: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")
print("Columnas binarias auxiliares creadas: RainToday_bin, RainTomorrow_bin")

Dimensiones tras limpieza básica: 145,460 filas x 25 columnas
Columnas binarias auxiliares creadas: RainToday_bin, RainTomorrow_bin


In [6]:
# Reporte inicial de calidad de datos
quality_report = pd.DataFrame({
    "métrica": [
        "filas originales",
        "columnas originales",
        "duplicados exactos",
        "filas luego de limpieza",
        "fechas inválidas",
        "variables con algún faltante",
    ],
    "valor": [
        len(df_raw),
        df_raw.shape[1],
        duplicados,
        len(df),
        df["Date"].isna().sum(),
        int((df_raw.isna().sum() > 0).sum()),
    ],
    "detalle": [
        "Registros cargados",
        "Variables del archivo original",
        f"{porc_duplicados:.3f}% de las filas",
        "No se eliminaron filas porque no había duplicados",
        "Conversión con errors='coerce'",
        f"{(df_raw.isna().sum() > 0).mean() * 100:.2f}% de las variables",
    ],
})
quality_report


,métrica,valor,detalle
0,filas originales,145460,Registros cargados
1,columnas originales,23,Variables del archivo original
2,duplicados exactos,0,0.000% de las filas
3,filas luego de limpieza,145460,No se eliminaron filas porque no había duplicados
4,fechas inválidas,0,Conversión con errors='coerce'
5,variables con algún faltante,21,91.30% de las variables


### 1.1 Control de inconsistencias de dominio

Además de los faltantes y duplicados, se revisan reglas básicas de coherencia: precipitaciones y velocidades no negativas, humedades en el rango 0–100, escalas de nubosidad en 0–9, temperatura máxima no inferior a la mínima y categorías binarias válidas. Estos controles no reemplazan una validación meteorológica especializada, pero permiten detectar errores evidentes de captura.

In [7]:
# Reglas básicas de consistencia
speed_cols = ["WindGustSpeed", "WindSpeed9am", "WindSpeed3pm"]

inconsistency_checks = {
    "Rainfall negativo": int((df["Rainfall"] < 0).sum()),
    "Humidity9am fuera de [0, 100]": int(((df["Humidity9am"] < 0) | (df["Humidity9am"] > 100)).sum()),
    "Humidity3pm fuera de [0, 100]": int(((df["Humidity3pm"] < 0) | (df["Humidity3pm"] > 100)).sum()),
    "Cloud9am fuera de [0, 9]": int(((df["Cloud9am"] < 0) | (df["Cloud9am"] > 9)).sum()),
    "Cloud3pm fuera de [0, 9]": int(((df["Cloud3pm"] < 0) | (df["Cloud3pm"] > 9)).sum()),
    "MaxTemp menor que MinTemp": int((df["MaxTemp"] < df["MinTemp"]).sum()),
    "Alguna velocidad de viento negativa": int((df[speed_cols] < 0).any(axis=1).sum()),
    "RainToday distinto de Yes/No": int((~df["RainToday"].dropna().isin(["No", "Yes"])).sum()),
    "RainTomorrow distinto de Yes/No": int((~df["RainTomorrow"].dropna().isin(["No", "Yes"])).sum()),
}

inconsistency_report = (
    pd.Series(inconsistency_checks, name="registros_inconsistentes")
    .rename_axis("control")
    .reset_index()
)
inconsistency_report["resultado"] = np.where(
    inconsistency_report["registros_inconsistentes"].eq(0),
    "Sin inconsistencias",
    "Revisar"
)
inconsistency_report


,control,registros_inconsistentes,resultado
0,Rainfall negativo,0,Sin inconsistencias
1,"Humidity9am fuera de [0, 100]",0,Sin inconsistencias
2,"Humidity3pm fuera de [0, 100]",0,Sin inconsistencias
3,"Cloud9am fuera de [0, 9]",0,Sin inconsistencias
4,"Cloud3pm fuera de [0, 9]",0,Sin inconsistencias
5,MaxTemp menor que MinTemp,0,Sin inconsistencias
6,Alguna velocidad de viento negativa,0,Sin inconsistencias
7,RainToday distinto de Yes/No,0,Sin inconsistencias
8,RainTomorrow distinto de Yes/No,0,Sin inconsistencias


### 1.2 Clasificación metodológica de variables

El feedback de la Formativa 1 solicitó completar la clasificación de variables, incorporando explícitamente variables discretas y ordinales. En WeatherAUS las variables de nubosidad (`Cloud9am`, `Cloud3pm`) pueden tratarse como **ordinales discretas**, ya que representan una escala ordenada de cobertura nubosa.

In [8]:
clasificacion = {
    "Cuantitativas continuas": [
        "MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
        "WindGustSpeed", "WindSpeed9am", "WindSpeed3pm",
        "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm",
        "Temp9am", "Temp3pm"
    ],
    "Ordinales discretas": [
        "Cloud9am", "Cloud3pm"
    ],
    "Categóricas nominales": [
        "Location", "WindGustDir", "WindDir9am", "WindDir3pm"
    ],
    "Binarias": [
        "RainToday", "RainTomorrow"
    ],
    "Temporal": ["Date"]
}

clasificacion_df = pd.DataFrame([
    {"tipo_variable": tipo, "variables": ", ".join(vars_)}
    for tipo, vars_ in clasificacion.items()
])
clasificacion_df


,tipo_variable,variables
0,Cuantitativas continuas,"MinTemp, MaxTemp, Rainfall, Evaporation, Sunsh..."
1,Ordinales discretas,"Cloud9am, Cloud3pm"
2,Categóricas nominales,"Location, WindGustDir, WindDir9am, WindDir3pm"
3,Binarias,"RainToday, RainTomorrow"
4,Temporal,Date


## 2. Análisis exploratorio de datos

Se calculan medidas descriptivas para variables numéricas, medidas de forma y frecuencias para variables categóricas y binarias. También se agregan visualizaciones y análisis bivariado para identificar relaciones relevantes con `RainTomorrow`.

In [9]:
# Variables numéricas principales para el informe
num_vars = [
    "MinTemp", "MaxTemp", "Rainfall", "Humidity9am", "Humidity3pm",
    "Pressure9am", "Temp9am", "Temp3pm", "WindGustSpeed"
]

# Tabla descriptiva ampliada: tendencia central, dispersión, posición, asimetría y curtosis
summary = df[num_vars].agg(['count', 'mean', 'std', 'min', 'median', 'max']).T
summary['q1'] = df[num_vars].quantile(0.25)
summary['q3'] = df[num_vars].quantile(0.75)
summary['iqr'] = summary['q3'] - summary['q1']
summary['cv'] = summary['std'] / summary['mean'].replace(0, np.nan)
summary['skew'] = df[num_vars].skew(numeric_only=True)
summary['kurtosis'] = df[num_vars].kurt(numeric_only=True)

summary = summary[['count', 'mean', 'std', 'min', 'q1', 'median', 'q3', 'max', 'iqr', 'cv', 'skew', 'kurtosis']]
summary_rounded = summary.round(2)
summary_rounded

,count,mean,std,min,q1,median,q3,max,iqr,cv,skew,kurtosis
MinTemp,143975.0,12.19,6.40,-8.5,7.6,12.0,16.9,33.9,9.3,0.52,0.02,-0.48
MaxTemp,144199.0,23.22,7.12,-4.8,17.9,22.6,28.2,48.1,10.3,0.31,0.22,-0.22
Rainfall,142199.0,2.36,8.48,0.0,0.0,0.0,0.8,371.0,0.8,3.59,9.84,178.15
Humidity9am,142806.0,68.88,19.03,0.0,57.0,70.0,83.0,100.0,26.0,0.28,-0.48,-0.04
Humidity3pm,140953.0,51.54,20.80,0.0,37.0,52.0,66.0,100.0,29.0,0.40,0.03,-0.51
Pressure9am,130395.0,1017.65,7.11,980.5,1012.9,1017.6,1022.4,1041.0,9.5,0.01,-0.10,0.23
Temp9am,143693.0,16.99,6.49,-7.2,12.3,16.7,21.6,40.2,9.3,0.38,0.09,-0.34
Temp3pm,141851.0,21.68,6.94,-5.4,16.6,21.1,26.4,46.7,9.8,0.32,0.24,-0.14
WindGustSpeed,135197.0,40.04,13.61,6.0,31.0,39.0,48.0,135.0,17.0,0.34,0.87,1.42


In [10]:
# Interpretación automática de forma para facilitar redacción del informe
def interpretar_asimetria(sk):
    if pd.isna(sk):
        return "sin dato"
    if abs(sk) < 0.5:
        return "aprox. simétrica"
    if sk >= 1:
        return "asimetría positiva alta"
    if sk <= -1:
        return "asimetría negativa alta"
    return "asimetría positiva moderada" if sk > 0 else "asimetría negativa moderada"

def interpretar_curtosis(ku):
    if pd.isna(ku):
        return "sin dato"
    if abs(ku) < 0.5:
        return "similar a normal"
    if ku > 0.5:
        return "colas pesadas / valores extremos"
    return "colas ligeras"

shape_table = pd.DataFrame({
    "variable": num_vars,
    "asimetría": [summary.loc[v, 'skew'] for v in num_vars],
    "curtosis": [summary.loc[v, 'kurtosis'] for v in num_vars],
})
shape_table["interpretación_asimetría"] = shape_table["asimetría"].apply(interpretar_asimetria)
shape_table["interpretación_curtosis"] = shape_table["curtosis"].apply(interpretar_curtosis)
shape_table.round(2)

,variable,asimetría,curtosis,interpretación_asimetría,interpretación_curtosis
0,MinTemp,0.02,-0.48,aprox. simétrica,similar a normal
1,MaxTemp,0.22,-0.22,aprox. simétrica,similar a normal
2,Rainfall,9.84,178.15,asimetría positiva alta,colas pesadas / valores extremos
3,Humidity9am,-0.48,-0.04,aprox. simétrica,similar a normal
4,Humidity3pm,0.03,-0.51,aprox. simétrica,colas ligeras
5,Pressure9am,-0.10,0.23,aprox. simétrica,similar a normal
6,Temp9am,0.09,-0.34,aprox. simétrica,similar a normal
7,Temp3pm,0.24,-0.14,aprox. simétrica,similar a normal
8,WindGustSpeed,0.87,1.42,asimetría positiva moderada,colas pesadas / valores extremos


In [11]:
# Frecuencias absolutas y relativas de variables categóricas y binarias
cat_vars = ["RainToday", "RainTomorrow", "Location", "WindGustDir", "WindDir9am", "WindDir3pm"]
frecuencias = {}

for var in cat_vars:
    counts = df[var].value_counts(dropna=False)
    n_valid = int(df[var].notna().sum())
    freq = counts.rename("frecuencia").to_frame()
    freq["% total"] = (freq["frecuencia"] / len(df) * 100).round(2)
    # Los faltantes no forman parte del denominador de % válido.
    valid_counts = df[var].dropna().value_counts()
    freq["% válido"] = np.nan
    for category, count in valid_counts.items():
        freq.loc[category, "% válido"] = round(count / n_valid * 100, 2)
    frecuencias[var] = freq

print("Frecuencia de RainTomorrow:")
display(frecuencias["RainTomorrow"])

print("Top 10 estaciones por número de registros:")
display(frecuencias["Location"].head(10))

print("Dirección de ráfaga de viento:")
display(frecuencias["WindGustDir"])


Frecuencia de RainTomorrow:


,frecuencia,% total,% válido
RainTomorrow,,,
No,110316,75.84,77.58
Yes,31877,21.91,22.42
NaN,3267,2.25,NaN


Top 10 estaciones por número de registros:


,frecuencia,% total,% válido
Location,,,
Canberra,3436,2.36,2.36
Sydney,3344,2.30,2.30
Darwin,3193,2.20,2.20
Melbourne,3193,2.20,2.20
Brisbane,3193,2.20,2.20
Adelaide,3193,2.20,2.20
Perth,3193,2.20,2.20
Hobart,3193,2.20,2.20
Albany,3040,2.09,2.09


Dirección de ráfaga de viento:


,frecuencia,% total,% válido
WindGustDir,,,
NaN,10326,7.10,NaN
W,9915,6.82,7.34
SE,9418,6.47,6.97
N,9313,6.40,6.89
SSE,9216,6.34,6.82
E,9181,6.31,6.79
S,9168,6.30,6.78
WSW,9069,6.23,6.71
SW,8967,6.16,6.64


In [12]:
# Carpeta de salida para las figuras del informe.
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Figuras se guardarán en: results/figures")

Figuras se guardarán en: results/figures


In [13]:
# Figura 1: histogramas de MaxTemp y Rainfall recortado al p99 para visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

max_temp = df["MaxTemp"].dropna()
axes[0].hist(max_temp, bins=40, edgecolor="black", alpha=0.75)
axes[0].axvline(max_temp.mean(), linestyle="--", linewidth=2, label=f"Media = {max_temp.mean():.2f}°C")
axes[0].set_title("Distribución de MaxTemp")
axes[0].set_xlabel("Temperatura máxima (°C)")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

rainfall_p99 = df["Rainfall"].dropna()
rainfall_p99 = rainfall_p99[rainfall_p99 <= rainfall_p99.quantile(0.99)]
axes[1].hist(rainfall_p99, bins=40, edgecolor="black", alpha=0.75)
axes[1].set_title("Distribución de Rainfall (≤ p99)")
axes[1].set_xlabel("Precipitación diaria (mm)")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
fig_path = FIG_DIR / "fig01_histogramas_maxtemp_rainfall.png"
plt.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print(fig_path.relative_to(PROJECT_ROOT))

results/figures/fig01_histogramas_maxtemp_rainfall.png


In [14]:
# Figura 2: boxplots individuales según RainTomorrow
valid_box = df[df["RainTomorrow"].isin(["No", "Yes"])]

boxplots = [
    ("MaxTemp", "Temperatura máxima (°C)", "fig02a_boxplot_maxtemp_rainTomorrow.png"),
    ("Humidity3pm", "Humedad 15:00 (%)", "fig02b_boxplot_humidity3pm_rainTomorrow.png"),
]

for variable, ylabel, filename in boxplots:
    fig, ax = plt.subplots(figsize=(8, 5))
    valid_box.boxplot(column=variable, by="RainTomorrow", ax=ax, grid=True)
    ax.set_title(f"{variable} según RainTomorrow")
    ax.set_xlabel("¿Llueve mañana?")
    ax.set_ylabel(ylabel)
    plt.suptitle("")
    plt.tight_layout()
    fig_path = FIG_DIR / filename
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(fig_path.relative_to(PROJECT_ROOT))

results/figures/fig02a_boxplot_maxtemp_rainTomorrow.png


results/figures/fig02b_boxplot_humidity3pm_rainTomorrow.png


In [15]:
# Figura 3: frecuencias relativas de variables categóricas
rain_counts = df["RainTomorrow"].dropna().value_counts(normalize=True).reindex(["No", "Yes"]) * 100
top_locations = df["Location"].value_counts().head(10).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(rain_counts.index.astype(str), rain_counts.values)
for pos, value in enumerate(rain_counts.values):
    axes[0].text(pos, value + 1, f"{value:.1f}%", ha="center", fontweight="bold")
axes[0].set_title("Distribución válida de RainTomorrow")
axes[0].set_xlabel("¿Llueve mañana?")
axes[0].set_ylabel("Porcentaje válido (%)")
axes[0].set_ylim(0, 100)

axes[1].barh(top_locations.index, top_locations.values)
axes[1].set_title("10 estaciones con más observaciones")
axes[1].set_xlabel("Frecuencia absoluta")
axes[1].set_ylabel("Location")

plt.tight_layout()
fig_path = FIG_DIR / "fig03_frecuencias_categoricas.png"
plt.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print(fig_path.relative_to(PROJECT_ROOT))


results/figures/fig03_frecuencias_categoricas.png


In [16]:
# Figura 4: matriz de correlación y asociaciones lineales con la variable objetivo
corr_vars = num_vars + ["RainTomorrow_bin", "RainToday_bin"]
corr = df[corr_vars].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
for row in range(len(corr.index)):
    for col in range(len(corr.columns)):
        value = corr.iloc[row, col]
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="Correlación de Pearson")
ax.set_title("Matriz de correlación entre variables meteorológicas")
plt.tight_layout()

fig_path = FIG_DIR / "fig04_matriz_correlacion.png"
plt.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print(fig_path.relative_to(PROJECT_ROOT))

target_correlations = (
    corr["RainTomorrow_bin"]
    .drop(labels=["RainTomorrow_bin"])
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .rename("correlación_con_RainTomorrow")
    .to_frame()
)
display(target_correlations.round(3))


results/figures/fig04_matriz_correlacion.png


,correlación_con_RainTomorrow
Humidity3pm,0.446
RainToday_bin,0.313
Humidity9am,0.257
Pressure9am,-0.246
Rainfall,0.239
WindGustSpeed,0.234
Temp3pm,-0.192
MaxTemp,-0.159
MinTemp,0.084
Temp9am,-0.026


In [17]:
# Análisis bivariado con la variable objetivo: medias por grupo
bivariado = df[df["RainTomorrow"].isin(["No", "Yes"])].groupby("RainTomorrow")[num_vars].agg(['count', 'mean', 'std', 'median'])
bivariado.round(2)

MinTemp                     MaxTemp                     Rainfall  \
               count   mean   std median   count   mean   std median    count   
RainTomorrow                                                                    
No            109854  11.90  6.40   11.8  110049  23.84  7.06   23.3   109586   
Yes            31702  13.19  6.32   12.7   31822  21.12  6.91   20.2    31201   

                    ... Temp9am        Temp3pm                     \
              mean  ...     std median   count  mean   std median   
RainTomorrow        ...                                             
No            1.27  ...    6.52   16.9  108332  22.4  6.85   21.9   
Yes           6.14  ...    6.38   16.1   31135  19.2  6.66   18.4   

             WindGustSpeed                       
                     count   mean    std median  
RainTomorrow                                     
No                  103488  38.29  12.41   37.0  
Yes                  29435  45.95  15.72   44.0  

[2 rows x 36 columns]

## 3. Estimación de parámetros

La Sumativa 1 solicita estimar parámetros para al menos tres variables numéricas relevantes. Se estiman intervalos de confianza al 95% para la media de `MaxTemp`, `Humidity3pm` y `WindGustSpeed` mediante distribución t de Student, porque la desviación estándar poblacional es desconocida. Además, se estima la proporción de días con lluvia al día siguiente mediante intervalo de Wilson, tal como fue sugerido en el feedback.

La interpretación del 95% se realiza como propiedad del procedimiento: si este método de construcción se repitiera muchas veces con muestras comparables, aproximadamente el 95% de los intervalos construidos contendría el verdadero parámetro poblacional.

In [18]:
def t_confidence_interval_mean(series, confidence=0.95):
    x = series.dropna().astype(float)
    n = len(x)
    mean = x.mean()
    sd = x.std(ddof=1)
    se = sd / math.sqrt(n)
    alpha = 1 - confidence
    tcrit = stats.t.ppf(1 - alpha/2, df=n-1)
    margin = tcrit * se
    return {
        "n": n,
        "estimación": mean,
        "desv_est": sd,
        "error_estándar": se,
        "IC95_inf": mean - margin,
        "IC95_sup": mean + margin,
        "método": "t de Student"
    }

def wilson_interval(successes, n, confidence=0.95):
    alpha = 1 - confidence
    z = stats.norm.ppf(1 - alpha/2)
    phat = successes / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2*n)) / denom
    half_width = z * math.sqrt((phat*(1-phat)/n) + (z**2/(4*n**2))) / denom
    return center - half_width, center + half_width

estimate_vars = ["MaxTemp", "Humidity3pm", "WindGustSpeed"]
ci_rows = []
for var in estimate_vars:
    res = t_confidence_interval_mean(df[var])
    ci_rows.append({"parámetro": f"Media de {var}", **res})

rain_valid = df["RainTomorrow"].dropna()
n_rain = len(rain_valid)
success_rain = (rain_valid == "Yes").sum()
phat_rain = success_rain / n_rain
wilson_inf, wilson_sup = wilson_interval(success_rain, n_rain)
ci_rows.append({
    "parámetro": "Proporción RainTomorrow = Yes",
    "n": n_rain,
    "estimación": phat_rain,
    "desv_est": np.nan,
    "error_estándar": np.nan,
    "IC95_inf": wilson_inf,
    "IC95_sup": wilson_sup,
    "método": "Wilson"
})

ci_table = pd.DataFrame(ci_rows)
ci_table_display = ci_table.copy()
for col in ["estimación", "desv_est", "error_estándar", "IC95_inf", "IC95_sup"]:
    ci_table_display[col] = ci_table_display[col].astype(float).round(4)
ci_table_display

,parámetro,n,estimación,desv_est,error_estándar,IC95_inf,IC95_sup,método
0,Media de MaxTemp,144199,23.2213,7.1190,0.0187,23.1846,23.2581,t de Student
1,Media de Humidity3pm,140953,51.5391,20.7959,0.0554,51.4306,51.6477,t de Student
2,Media de WindGustSpeed,135197,40.0352,13.6071,0.0370,39.9627,40.1078,t de Student
3,Proporción RainTomorrow = Yes,142193,0.2242,NaN,NaN,0.2220,0.2264,Wilson


In [19]:
# Figura 5: intervalos de confianza para medias numéricas seleccionadas
ci_means = ci_table[ci_table["método"] == "t de Student"].copy()
labels = ci_means["parámetro"].str.replace("Media de ", "", regex=False)
est = ci_means["estimación"].values
err_low = est - ci_means["IC95_inf"].values
err_high = ci_means["IC95_sup"].values - est

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(labels, est, yerr=[err_low, err_high], fmt='o', capsize=5)
ax.set_title("IC 95% para medias de variables seleccionadas")
ax.set_ylabel("Estimación puntual e IC 95%")
ax.set_xlabel("Variable")
plt.tight_layout()
fig_path = FIG_DIR / "fig05_intervalos_confianza_medias.png"
plt.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print(fig_path.relative_to(PROJECT_ROOT))

results/figures/fig05_intervalos_confianza_medias.png


## 4. Pruebas de hipótesis

Se reemplaza la prueba de `MaxTemp` contra 22 °C, observada por el profesor como poco conectada a una decisión, por una comparación entre días con y sin lluvia al día siguiente. Ambas pruebas responden a una decisión concreta: identificar variables meteorológicas que puedan priorizarse para anticipar `RainTomorrow`.

Se usa la prueba t de Welch porque compara medias entre dos grupos independientes sin exigir igualdad de varianzas. Debido al gran tamaño muestral, el Teorema Central del Límite respalda la aproximación de la distribución muestral de las medias, aunque las variables originales no sean perfectamente normales. Se reporta además el tamaño del efecto mediante d de Cohen, para distinguir significancia estadística de relevancia práctica.

In [20]:
def cohens_d_independent(x, y):
    x = pd.Series(x).dropna().astype(float)
    y = pd.Series(y).dropna().astype(float)
    nx, ny = len(x), len(y)
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    pooled_sd = math.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    return (x.mean() - y.mean()) / pooled_sd

def welch_test_by_rain(var):
    data = df[[var, "RainTomorrow"]].dropna()
    no = data.loc[data["RainTomorrow"] == "No", var]
    yes = data.loc[data["RainTomorrow"] == "Yes", var]
    t_stat, p_value = stats.ttest_ind(yes, no, equal_var=False, nan_policy='omit')
    d = cohens_d_independent(yes, no)  # Yes - No
    return {
        "variable": var,
        "n_yes": len(yes),
        "media_yes": yes.mean(),
        "sd_yes": yes.std(ddof=1),
        "n_no": len(no),
        "media_no": no.mean(),
        "sd_no": no.std(ddof=1),
        "diferencia_yes_no": yes.mean() - no.mean(),
        "t_welch": t_stat,
        "p_valor": p_value,
        "cohens_d": d
    }

hyp_vars = ["MaxTemp", "Humidity3pm"]
hyp_table = pd.DataFrame([welch_test_by_rain(v) for v in hyp_vars])
hyp_table.round(4)

,variable,n_yes,media_yes,sd_yes,n_no,media_no,sd_no,diferencia_yes_no,t_welch,p_valor,cohens_d
0,MaxTemp,31822,21.1191,6.9115,110049,23.8362,7.0598,-2.7171,-61.4679,0.0,-0.3867
1,Humidity3pm,30913,68.8000,19.0374,107670,46.5106,18.4895,22.2894,182.6077,0.0,1.1975


In [21]:
# Resumen de supuestos, tamaños del efecto y decisiones
alpha = 0.05
hyp_interpretation = hyp_table.copy()
hyp_interpretation["p_valor_mostrado"] = hyp_interpretation["p_valor"].apply(
    lambda p: "< 0.001" if p < 0.001 else f"{p:.4f}"
)
hyp_interpretation["decisión"] = np.where(
    hyp_interpretation["p_valor"] < alpha,
    "Rechazar H0",
    "No rechazar H0"
)
hyp_interpretation["interpretación_efecto"] = hyp_interpretation["cohens_d"].abs().apply(
    lambda d: "muy pequeño" if d < 0.2 else (
        "pequeño" if d < 0.5 else (
            "moderado" if d < 0.8 else "grande"
        )
    )
)

hyp_interpretation[
    [
        "variable", "diferencia_yes_no", "t_welch",
        "p_valor_mostrado", "cohens_d",
        "interpretación_efecto", "decisión"
    ]
].round({"diferencia_yes_no": 4, "t_welch": 4, "cohens_d": 4})


,variable,diferencia_yes_no,t_welch,p_valor_mostrado,cohens_d,interpretación_efecto,decisión
0,MaxTemp,-2.7171,-61.4679,< 0.001,-0.3867,pequeño,Rechazar H0
1,Humidity3pm,22.2894,182.6077,< 0.001,1.1975,grande,Rechazar H0


### Hipótesis evaluadas

**Prueba 1 — MaxTemp según RainTomorrow**

- H0: la media de `MaxTemp` es igual entre días con `RainTomorrow = Yes` y `RainTomorrow = No`.
- H1: la media de `MaxTemp` difiere entre ambos grupos.
- Decisión asociada: evaluar si la temperatura máxima aporta evidencia útil para distinguir días previos a lluvia futura.

**Prueba 2 — Humidity3pm según RainTomorrow**

- H0: la media de `Humidity3pm` es igual entre días con `RainTomorrow = Yes` y `RainTomorrow = No`.
- H1: la media de `Humidity3pm` difiere entre ambos grupos.
- Decisión asociada: evaluar si la humedad vespertina debe priorizarse como variable explicativa o predictora de lluvia futura.

## 5. Informe de avance del proyecto: interpretación integrada

Los resultados descriptivos muestran que `RainTomorrow` está desbalanceada: cerca de tres cuartas partes de los registros válidos corresponden a días sin lluvia al día siguiente, mientras que la clase positiva es minoritaria. Esto es importante para futuras decisiones de modelamiento, porque una métrica como exactitud podría sobrevalorar un modelo que prediga mayoritariamente la clase `No`.

La variable `Rainfall` presenta una asimetría positiva muy marcada y curtosis elevada, lo que confirma que la mayoría de los días registra lluvia nula o baja, mientras que pocos eventos concentran precipitaciones altas. Por ello, en fases posteriores convendrá evaluar transformaciones, tratamiento de valores extremos o segmentación de eventos de lluvia intensa.

Los intervalos de confianza para `MaxTemp`, `Humidity3pm` y `WindGustSpeed` permiten cuantificar la incertidumbre sobre parámetros meteorológicos promedio. La proporción de `RainTomorrow = Yes`, estimada mediante Wilson, entrega una línea base para la ocurrencia de lluvia futura. La interpretación correcta del 95% corresponde al procedimiento de construcción del intervalo, no a una probabilidad posterior sobre un intervalo ya calculado.

Las pruebas de hipótesis sugieren que tanto `MaxTemp` como `Humidity3pm` difieren entre días con y sin lluvia al día siguiente. Sin embargo, la interpretación no debe basarse solo en el valor p, porque con muestras grandes diferencias pequeñas pueden resultar estadísticamente significativas. Por eso se reporta el tamaño del efecto. En particular, `Humidity3pm` presenta una diferencia práctica más relevante y queda como una variable prioritaria para apoyar decisiones de anticipación de lluvia.

### Próximos pasos

- Profundizar el tratamiento de valores faltantes, especialmente en variables como `Sunshine`, `Evaporation`, `Cloud9am` y `Cloud3pm`.
- Evaluar transformaciones o tratamiento robusto para variables altamente asimétricas como `Rainfall`.
- Seleccionar variables relevantes para un modelo predictivo de `RainTomorrow`.
- Considerar estrategias de evaluación adecuadas para clases desbalanceadas, como recall, precisión, F1-score y matriz de confusión.
- Mantener el notebook como fuente reproducible para las tablas, figuras e interpretaciones del informe técnico.

## 6. Notebook y repositorio GitHub

Este notebook sigue el orden de la Sumativa 1:

1. Preparación y carga de datos.
2. Análisis exploratorio de datos.
3. Estimación de parámetros.
4. Pruebas de hipótesis.
5. Interpretación preliminar y próximos pasos.

La semilla de reproducibilidad es:

```python
SEED = 42
```

**Verificación final:** esta versión fue ejecutada de principio a fin con **Kernel → Restart & Run All**, sin errores ni intervención manual. Las tablas, pruebas y figuras del informe provienen de esta misma ejecución.

Las figuras se guardan automáticamente en `results/figures`. El archivo `weatherAUS.csv` debe conservarse en `data/raw/` para mantener la estructura reproducible del repositorio.


## 7. Bibliografía base para el informe

- Kaggle. (2017). *Rain in Australia dataset*. https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package
- Maidana, J. P. (2026). *Medidas de dispersión y forma: variabilidad, asimetría y curtosis* [Apunte de curso]. Universidad Andrés Bello.
- Maidana, J. P. (2026). *Intervalos de confianza: cuantificando la incertidumbre estadística* [Apunte de curso]. Universidad Andrés Bello.
- Maidana, J. P. (2026). *Pruebas de hipótesis: pruebas t y Z para una muestra* [Apunte de curso]. Universidad Andrés Bello.
- McKinney, W. (2010). Data structures for statistical computing in Python. En S. van der Walt y J. Millman (Eds.), *Proceedings of the 9th Python in Science Conference* (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a
- Virtanen, P., Gommers, R., Oliphant, T. E., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods, 17*, 261–272. https://doi.org/10.1038/s41592-019-0686-2
